# Worked Capstone: Statistical Process Comparison

**Domain:** Experiment analysis and operational statistics  
**Primary dataset:** `synthetic experiment generated in notebook`  
**Level:** Practitioner to Advanced

## Business goal

Determine whether a redesigned process changes completion time by an operationally meaningful amount, while reporting uncertainty and assumptions.

This is a worked reference project. First attempt the corresponding phase project independently; then use this capstone to compare framing, evaluation, code structure, and communication.

## Decision questions

        1. What is the estimand?
2. How large and uncertain is the difference?
3. Are assumptions plausible?
4. Is the effect operationally meaningful?

        ## Definition of done

        - [ ] Design statement
- [ ] Descriptive statistics
- [ ] Confidence interval
- [ ] Welch test
- [ ] Bootstrap sensitivity
- [ ] Practical recommendation

## End-to-end workflow

```text
Decision and scope
      ↓
Data contract and quality
      ↓
Exploration and hypotheses
      ↓
Baseline and evaluation design
      ↓
Candidate method(s)
      ↓
Held-out / temporal evaluation
      ↓
Error, slice, and sensitivity analysis
      ↓
Artifacts, limitations, recommendation
```

At every stage, distinguish calculation correctness, statistical validity, operational validity, and decision validity.

## Risk register

        | Risk | Mitigation |
        |---|---|
        | Non-random assignment | Treat result as associational or redesign the experiment. |
| Skew/outliers | Use robust summaries and bootstrap sensitivity. |
| Significance without value | Compare interval with a predeclared meaningful effect. |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## 1. Design and synthetic data

The estimand is the difference in population mean completion time under redesign versus control. Lower is better.

In [ ]:
from scipy import stats
rng_local=np.random.default_rng(42)
control=rng_local.lognormal(np.log(44),.30,160)
redesign=rng_local.lognormal(np.log(40),.29,158)
minimum_meaningful_reduction=-3.0
data=pd.DataFrame({
    "group":np.repeat(["control","redesign"],[len(control),len(redesign)]),
    "completion_minutes":np.r_[control,redesign],
})
display(data.groupby("group").completion_minutes.agg(["count","mean","median","std"]).round(3))

## 2. Distribution and robust comparison

Visualize distributions and calculate both mean and median differences.

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
ax.hist([control,redesign],bins=30,label=["Control","Redesign"],alpha=.6)
ax.set(title="Completion-time distributions",xlabel="Minutes",ylabel="Cases")
ax.legend(); plt.show()
print("Mean difference:",redesign.mean()-control.mean())
print("Median difference:",np.median(redesign)-np.median(control))

## 3. Inference and effect size

Use Welch's test for unequal variances and report a standardized effect.

In [ ]:
t_stat,p_value=stats.ttest_ind(redesign,control,equal_var=False)
difference=redesign.mean()-control.mean()
se=np.sqrt(redesign.var(ddof=1)/len(redesign)+control.var(ddof=1)/len(control))
df_w=(redesign.var(ddof=1)/len(redesign)+control.var(ddof=1)/len(control))**2/(
    (redesign.var(ddof=1)/len(redesign))**2/(len(redesign)-1)+
    (control.var(ddof=1)/len(control))**2/(len(control)-1)
)
critical=stats.t.ppf(.975,df_w)
ci=(difference-critical*se,difference+critical*se)
pooled=np.sqrt(((len(redesign)-1)*redesign.var(ddof=1)+(len(control)-1)*control.var(ddof=1))/
               (len(redesign)+len(control)-2))
print({"difference":difference,"95% CI":ci,"t":t_stat,"p":p_value,"Cohen_d":difference/pooled})

## 4. Bootstrap sensitivity

Bootstrap the statistic to reduce reliance on a closed-form Normal approximation.

In [ ]:
boot=np.empty(6000)
for i in range(len(boot)):
    boot[i]=rng_local.choice(redesign,len(redesign),replace=True).mean()-rng_local.choice(control,len(control),replace=True).mean()
boot_ci=np.quantile(boot,[.025,.975])
print("Bootstrap 95% interval:",boot_ci)
print("P(reduction exceeds meaningful threshold) approx:",np.mean(boot<=minimum_meaningful_reduction))

## 5. Decision statement

Translate the interval into a practical conclusion and name the design assumptions.

In [ ]:
decision={
    "estimand":"Mean minutes under redesign minus mean minutes under control",
    "estimate":float(difference),
    "interval":[float(ci[0]),float(ci[1])],
    "meaningful_threshold":minimum_meaningful_reduction,
    "recommendation":"Pilot with outcome and quality monitoring; evidence suggests a reduction, but validate randomization and downstream quality.",
}
print(json.dumps(decision,indent=2))
(Path(ARTIFACT_DIR)/"capstone_statistical_decision.json").write_text(json.dumps(decision,indent=2))

## Model/project card

Complete this before presenting the result:

| Field | Statement |
|---|---|
| Intended use | |
| Excluded use | |
| Data population and coverage | |
| Target/metric definition | |
| Evaluation split | |
| Baseline | |
| Primary result | |
| Known limitations | |
| Important subgroup behaviour | |
| Human review / abstention | |
| Monitoring | |
| Owner and review cadence | |

## Final reflection

1. Which result changed your initial belief?
2. Which assumption creates the largest residual risk?
3. What simpler alternative was competitive?
4. What evidence is still required before an operational decision?
5. What would you monitor first after release?

Re-run the notebook from a clean kernel and verify generated artifacts before considering the capstone complete.